In [1]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV


In [2]:
df=pd.read_csv("../data/Preprocessed_data.csv")

In [3]:
X=df.drop('accident_risk',axis=1)
y=df['accident_risk']

In [4]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.05,random_state=15,shuffle=True)

In [5]:
cols_to_scale=["speed_limit"]
Scaler=StandardScaler()

In [6]:
X_train[cols_to_scale] = Scaler.fit_transform(X_train[cols_to_scale])

In [7]:
X_test[cols_to_scale]= Scaler.transform(X_test[cols_to_scale])

In [8]:
models = {
    "Linear Regression": LinearRegression(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0),
}

In [9]:
results = []

for name, model in models.items():
    print(f"🚀 Training {name}...")

    model.fit(X_train, y_train)

    # Predictions
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    # Metrics
    train_rmse = root_mean_squared_error(y_train, train_preds)
    test_rmse = root_mean_squared_error(y_test, test_preds)
    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)

    results.append({
        "Model": name,
        "Train RMSE": round(train_rmse, 3),
        "Test RMSE": round(test_rmse, 3),
        "Train R²": round(train_r2, 3),
        "Test R²": round(test_r2, 3)
    })

    print(f"✅ {name} done!")
    print(f"   🔹 Train → RMSE: {train_rmse:.3f}, R²: {train_r2:.3f}")
    print(f"   🔹 Test  → RMSE: {test_rmse:.3f}, R²: {test_r2:.3f}\n")


🚀 Training Linear Regression...
✅ Linear Regression done!
   🔹 Train → RMSE: 0.073, R²: 0.805
   🔹 Test  → RMSE: 0.074, R²: 0.804

🚀 Training Gradient Boosting...
✅ Gradient Boosting done!
   🔹 Train → RMSE: 0.057, R²: 0.883
   🔹 Test  → RMSE: 0.057, R²: 0.883

🚀 Training Hist Gradient Boosting...
✅ Hist Gradient Boosting done!
   🔹 Train → RMSE: 0.056, R²: 0.886
   🔹 Test  → RMSE: 0.056, R²: 0.886

🚀 Training AdaBoost...
✅ AdaBoost done!
   🔹 Train → RMSE: 0.087, R²: 0.726
   🔹 Test  → RMSE: 0.087, R²: 0.725

🚀 Training XGBoost...
✅ XGBoost done!
   🔹 Train → RMSE: 0.055, R²: 0.889
   🔹 Test  → RMSE: 0.056, R²: 0.887



In [10]:
results_df = pd.DataFrame(results).sort_values(by="Test RMSE")
print("\n📊 Final Results Summary:")
display(results_df.style.background_gradient(cmap="coolwarm"))



📊 Final Results Summary:


,Model,Train RMSE,Test RMSE,Train R²,Test R²
2,Hist Gradient Boosting,0.056000,0.056000,0.886000,0.886000
4,XGBoost,0.055000,0.056000,0.889000,0.887000
1,Gradient Boosting,0.057000,0.057000,0.883000,0.883000
0,Linear Regression,0.073000,0.074000,0.805000,0.804000
3,AdaBoost,0.087000,0.087000,0.726000,0.725000


In [11]:
model = HistGradientBoostingRegressor(random_state=0)
param_dist = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_iter": [100, 300, 500],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [20, 50, 100]
}

search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=12,
                            scoring='r2', cv=5, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
print("Best:", search.best_params_, "Best score:", search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/home/amar/anaconda3/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best: {'min_samples_leaf': 50, 'max_leaf_nodes': 63, 'max_iter': 500, 'learning_rate': 0.05} Best score: 0.8862336529826497


In [13]:
best_params = {
    'min_samples_leaf': 50,
    'max_leaf_nodes': 63,
    'max_iter': 300,
    'learning_rate': 0.05,
    'random_state': 0,
    'early_stopping': True,
    'validation_fraction': 0.1
}

model = HistGradientBoostingRegressor(**best_params)

print("\nTraining model with best params...")
model.fit(X_train, y_train)
print("Done training.")

# ===== 5) تقييم على Train و Test
def evaluate(name, y_true, y_pred):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{name}: RMSE={rmse:.4f}, MAE={mae:.4f}, MSE={mse:.4f}, R²={r2:.4f}")
    
    return {
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'MSE': mse,
        'R²': r2
    }

print("\nEvaluating...")
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_metrics = evaluate("HistGradientBoostingRegressor - Train", y_train, train_pred)
test_metrics  = evaluate("HistGradientBoostingRegressor - Test",  y_test,  test_pred)


Training model with best params...
Done training.

Evaluating...
HistGradientBoostingRegressor - Train: RMSE=0.0558, MAE=0.0434, MSE=0.0558, R²=0.8874
HistGradientBoostingRegressor - Test: RMSE=0.0561, MAE=0.0433, MSE=0.0561, R²=0.8868
